# 8-0(참고). ARIMA·GARCH 설명 (라이브러리 기반)

이 시리즈의 ARIMA(`pmdarima`)와 GARCH(`arch`)는 라이브러리를 그대로 쓴다. SV는 표준 라이브러리가 없어 직접 구현했다(`8-0(참고). SV 설명`).

## 1-1. ARIMA

AR(자기회귀) + I(차분) + MA(이동평균)이다.

```
평균식: y'_t = φ_1 y'_(t-1) + ... + φ_p y'_(t-p) + ε_t + θ_1 ε_(t-1) + ...   (y'는 d번 차분한 값)
```

- **p**: 몇 시차 전 값까지 참고, **d**: 차분 횟수, **q**: 몇 시차 전 오차까지 참고
- **오차 ε_t는 iid 정규분포 N(0,σ²)**가 기본 가정이다 -- CRPS를 정규분포 공식으로 채점하는 근거다.

## 1-2. `pmdarima`

```python
m = pm.auto_arima(train, seasonal=False)                      # (A) d는 KPSS/ADF, p,q는 AIC로 자동 탐색
m = pm.ARIMA(order=(1, 1, 0), with_intercept=False).fit(train)  # (B) 직접 지정
fc, conf = m.predict(n_periods=4, return_conf_int=True, alpha=0.10)
```

- 이 프로젝트는 (A) 대신 **(B)만** 쓴다. `auto_arima`의 `start_p`/`max_p`는 탐색 범위일 뿐 강제가 아니라서 원하는 p로 고정한 줄 알았는데 다른 값이 선택되는 문제가 있었다.
- **정확우도**(MLE)라 p,q를 바꿔도 유효표본수가 같아 AIC 비교가 안전하다.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import pmdarima as pm

raw = pd.read_csv("../../data/gold/gold_major_variables.csv")
raw = raw[raw["quarter"].notna()].reset_index(drop=True)
vals = raw[["quarter", "X3_l1"]].dropna()["X3_l1"].values.astype(float)
train = vals[:35]

m_auto = pm.auto_arima(train, seasonal=False, suppress_warnings=True)
print("auto_arima가 고른 order(p,d,q):", m_auto.order)
print("계수(순서대로):", np.round(m_auto.params(), 4))

m_fixed = pm.ARIMA(order=(1, 1, 0), with_intercept=False, suppress_warnings=True).fit(train)
fc, conf = m_fixed.predict(n_periods=4, return_conf_int=True, alpha=0.10)
print("\nARIMA(1,1,0) 4분기 예측:", np.round(fc, 1))
print("90% 구간:", np.round(conf, 1))


auto_arima가 고른 order(p,d,q): (2, 0, 1)
계수(순서대로): [ 3.1587800e+01  1.8643000e+00 -9.2720000e-01 -5.6720000e-01
  2.2251208e+03]

ARIMA(1,1,0) 4분기 예측: [833.7 856.  868.3 874.9]
90% 구간: [[ 750.4  917. ]
 [ 702.7 1009.4]
 [ 651.2 1085.3]
 [ 600.9 1148.9]]


## 1-3. GARCH

분산(변동성)이 시간에 따라 변한다고 보는 모델이다(ARIMA는 분산이 일정하다고 가정).

```
평균식:  y_t = φ y_(t-1) + ε_t
분산식:  σ_t² = ω + α ε_(t-1)² + β σ_(t-1)²
```

- **ω**: 변동성 바닥 수준, **α**: 직전 충격의 즉각 반영, **β**: 변동성의 지속성
- 평균식(φ)과 분산식(ω,α,β)을 **하나의 우도로 동시에 추정**한다 -- 이 프로젝트의 "동시최적화" 원형이다.

## 1-4. `arch`

```python
am = ARX(diff_train, lags=1, constant=False, rescale=False)   # p=0이면 ZeroMean
am.volatility = GARCH(p=1, q=1)
res = am.fit(disp="off"); fc = res.forecast(horizon=4, reindex=False)
```

- 평균식은 AR(p)만 지원(MA 없음)이라, 세 모델을 공정하게 비교하려고 ARIMA도 MA 없이 AR만 쓴다.
- 분산식은 GARCH(1,1)로 고정한다(대부분의 시계열에 충분한 경험칙).
- `constant=False`: 기본값은 상수항(Const)이 포함되어 AR단독과 평균식 조건이 달라진다. 계산 오류가 아니라 라이브러리 기본값이라 `8-2`~`8-5`는 이를 뺐다(`8-6(참고)`).
- **조건부우도**라 p를 늘리면 유효표본수가 줄어 AIC로 p를 비교할 수 없다(`8-3`, `8-4` 7절).

In [2]:
from arch import arch_model

diff_train = np.diff(train, n=1)
am = arch_model(diff_train, mean="AR", lags=1, vol="GARCH", p=1, q=1, rescale=False)
res = am.fit(disp="off")
print(res.params)

fcst = res.forecast(horizon=4, reindex=False)
print("\n분산 예측(h=1~4):", np.round(fcst.variance.values[-1], 1))


Const          4.599895
y[1]           0.563787
omega       1204.089886
alpha[1]       0.424004
beta[1]        0.059880
Name: params, dtype: float64

분산 예측(h=1~4): [1827.6 2669.3 3063.1 3249.3]


## 1-5. 두 라이브러리 모두 정규분포를 전제로 구간을 만든다

`pmdarima`는 오차가 iid 정규, `arch`는 기본 `dist="normal"`이다. 그래서 (평균, σ)를 `ps.crps_gaussian()`으로 채점하는 것이 맞고, 이 시리즈가 CRPS(정규분포)를 쓰는 근거다.

| | ARIMA | GARCH |
|---|---|---|
| 모델링 대상 | 평균(수준) | 분산(변동성) |
| 우도 | 정확우도 | 조건부우도 (AR차수 AIC 비교 무효) |
| 이 프로젝트에서 | d,p를 확정한 뒤 직접 fit | GARCH(1,1) 고정, AR평균과 동시추정 |

## 1-6. 파라미터는 어떻게 찾나

- **자동 추정(fit, MLE)**: ARIMA는 φ, GARCH는 AR계수·ω·α·β -- 훈련 데이터마다 새로 계산한다.
- **사람이 정함**: GARCH 분산식 차수 (1,1).
- **그리드서치로 정함**: 구조 파라미터 d, p뿐이다 (`8-4`).